# CNN-BiLSTM EEG ADHD Prediction Pipeline
**Dataset:** imsathish/eeg-adhd-nasrabadi-mat  
**Run all cells top to bottom. GPU T4 x1 must be enabled.**

In [ ]:
import os, shutil

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR = "/kaggle/input/datasets/imsathish/eeg-adhd-nasrabadi-mat"
OUT_DIR  = "/kaggle/working/results"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "figures"), exist_ok=True)

# ── Verify dataset is mounted ──────────────────────────────────────────────────
print("Checking dataset path ...")
if os.path.isdir(DATA_DIR):
    print(f"OK   {DATA_DIR} exists")
    mat_files = []
    for root, dirs, files in os.walk(DATA_DIR):
        for f in files:
            if f.endswith(".mat"):
                mat_files.append(os.path.join(root, f))
    print(f"     Found {len(mat_files)} .mat files")
    if len(mat_files) == 0:
        print("ERROR: No .mat files found.")
    elif len(mat_files) < 100:
        print(f"WARNING: Expected ~121 files, found {len(mat_files)}.")
    else:
        print("     Dataset looks correct — ready to run.")
    print("\n     Folder structure:")
    for item in sorted(os.listdir(DATA_DIR)):
        item_path = os.path.join(DATA_DIR, item)
        if os.path.isdir(item_path):
            n = sum(1 for f in os.listdir(item_path) if f.endswith(".mat"))
            print(f"       {item}/   ({n} .mat files)")
        else:
            print(f"       {item}")
else:
    print(f"ERROR: {DATA_DIR} not found.")
    print("Searching for .mat files elsewhere ...")
    for root, dirs, files in os.walk("/kaggle/input"):
        mat_here = [f for f in files if f.endswith(".mat")]
        if mat_here:
            print(f"  Found {len(mat_here)} .mat files at: {root}")

print(f"\nOUT_DIR = {OUT_DIR}")

In [ ]:
# Install only packages missing from Kaggle's environment
# Do NOT install tensorflow or numpy — Kaggle already has them and reinstalling breaks JAX
!pip install -q "shap>=0.44" mne
print("Packages ready.")

In [ ]:
# Clone pipeline code
if os.path.exists("/kaggle/working/project"):
    shutil.rmtree("/kaggle/working/project")

!git clone https://github.com/sathishshah/ADHD-Prediction.git /kaggle/working/project
print("Clone done.")

In [ ]:
# Verify TensorFlow and GPU
import tensorflow as tf
import numpy as np
import shap, mne, scipy, sklearn

print(f"TensorFlow  : {tf.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"SHAP        : {shap.__version__}")
print(f"MNE         : {mne.__version__}")

gpus = tf.config.list_physical_devices('GPU')
print(f"\nGPUs found  : {len(gpus)}")
for g in gpus:
    print(f"  {g}")
if not gpus:
    print("WARNING: No GPU detected — training will be very slow.")
    print("Go to Settings (right sidebar) > Accelerator > GPU T4 x1")

In [ ]:
# Smoke test — verifies full pipeline end-to-end in ~10 min
# Results from this run are NOT for the paper
!cd /kaggle/working/project && python -m adhd_pipeline.run_all \
    --data-dir {DATA_DIR} \
    --out-dir /kaggle/working/results_quick \
    --quick \
    --no-shap

In [ ]:
# Full pipeline run — ~90 min on T4 GPU
# --checkpoint saves each fold so a disconnect does not lose progress
!cd /kaggle/working/project && python -m adhd_pipeline.run_all \
    --data-dir {DATA_DIR} \
    --out-dir {OUT_DIR} \
    --checkpoint

In [ ]:
# Check all expected output files
import json

expected = [
    "all_results.json",
    "macro_replacements.tex",
    "folds.json",
    "shap_channel_importance.npy",
    "shap_band_importance.npy",
    "figures/leakage_bars.csv",
    "figures/baseline_bars.csv",
    "figures/roc_aucs.csv",
    "figures/confusion_matrix.csv",
    "figures/shap_channels.csv",
]

print("Output file check:\n")
all_ok = True
for fname in expected:
    path = os.path.join(OUT_DIR, fname)
    exists = os.path.isfile(path)
    print(f"  [{'OK     ' if exists else 'MISSING'}]  {fname}")
    if not exists:
        all_ok = False

print()
print("All files present — ready to download." if all_ok else
      "Some files missing — check the full run cell above for errors.")

# Print macro_replacements.tex
tex = os.path.join(OUT_DIR, "macro_replacements.tex")
if os.path.isfile(tex):
    print("\n" + "="*60)
    print("macro_replacements.tex — paste this into main.tex:")
    print("="*60)
    print(open(tex).read())

In [ ]:
# Print accuracy summary table
results_path = os.path.join(OUT_DIR, "all_results.json")
if not os.path.isfile(results_path):
    print("all_results.json not found — run the full pipeline cell first.")
else:
    results = json.load(open(results_path))
    model_order = ["LogisticRegression", "SVM", "RandomForest",
                   "CNN_only", "LSTM_only", "CNN_BiLSTM"]

    for proto, label in [
        ("protocol_a", "PROTOCOL A  Segment-wise (leaky)"),
        ("protocol_b", "PROTOCOL B  Subject-wise (clean)"),
        ("loso",       "PROTOCOL    LOSO"),
    ]:
        if proto not in results:
            continue
        print(f"\n{label}")
        print("-" * 52)
        print(f"  {'Model':<22}  {'Acc mean±std':>16}   AUC")
        print("-" * 52)
        for m in model_order:
            if m not in results[proto]:
                continue
            s = results[proto][m]["summary"]
            acc_m, acc_s = s["accuracy"]
            auc_m, _     = s["roc_auc"]
            print(f"  {m:<22}  {acc_m:>6.1f} ± {acc_s:<5.1f}  {auc_m:.3f}")

    sig = results.get("significance", {})
    if sig:
        print("\nLEAKAGE GAP  CNN-BiLSTM A vs B")
        print("-" * 36)
        for k, v in sig.items():
            print(f"  {k}: {v}")

In [ ]:
# Zip results for download
zip_path = "/kaggle/working/results_export"
shutil.make_archive(zip_path, "zip", OUT_DIR)
size_mb = os.path.getsize(zip_path + ".zip") / 1e6
print(f"Created: {zip_path}.zip  ({size_mb:.1f} MB)")
print("\nDownload steps:")
print("  1. Click the Output tab on the right sidebar")
print("  2. Download results_export.zip")
print("  3. Extract -> macro_replacements.tex and figures/*.csv are inside")